In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [21]:
data = pd.read_csv('Churn_Modelling.csv')

data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder()
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns = onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

# test_size=0.2 -> 20% of data → Test set, 80% → Train set
# random_state=42 -> Seed value → Ensures same split every time (reproducible results)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# We use this because neural networks and many ML algorithms work better when:
#  1. All features are on a similar scale
#  2. No feature dominates because of larger values
#  Example: Age: 20–60, Salary: 20,000–100,000
#  Salary has a much bigger scale → model becomes biased toward it Standardization fixes that imbalance.

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)
with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [22]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model
    

In [23]:
## Create a Keras classifier
# It is a wrapper that allows a Keras model to be used with scikit-learn tools such as:
# fit(), predict() GridSearchCV, cross_val_score
# So this classifier behaves like any sklearn model.
# verbose=1 Shows training progress (0 = silent, 1 = progress bar, 2 = one-line per epoch)
model = KerasClassifier(layers=1, neurons=32, build_fn=create_model, verbose=1)

In [24]:
# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [25]:
# Perform grid search
# n_jobs=-1 -> Use all CPU cores → faster training
# cv=3 -> 3-fold cross-validation
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


/opt/anaconda3/envs/py310/lib/python3.10/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/opt/anaconda3/envs/py310/lib/python3.10/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/opt/anaconda3/envs/py310/lib/python3.10/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/opt/anaconda3/envs/py310/lib/python3.10/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initi

Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50


2025-11-26 13:36:53.069149: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-11-26 13:36:53.074045: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-11-26 13:36:53.082234: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-11-26 13:36:53.084522: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-11-26 13:36:53.092065: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-11-26 13:36:53.100658: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-11-26 13:36:53.104875: I tensorflow/core/grappler/optimizers/cust

167/167 [==============================] - 3s 13ms/step - loss: 0.5531 - accuracy: 0.7291
Epoch 2/50
167/167 [==============================] - 3s 13ms/step - loss: 0.5286 - accuracy: 0.7480
Epoch 2/50
167/167 [==============================] - 3s 13ms/step - loss: 0.5313 - accuracy: 0.7537
Epoch 2/50
Epoch 2/50
167/167 [==============================] - 3s 13ms/step - loss: 0.5752 - accuracy: 0.7129
Epoch 2/50
167/167 [==============================] - 3s 13ms/step - loss: 0.6752 - accuracy: 0.6544
Epoch 2/50
167/167 [==============================] - 3s 13ms/step - loss: 0.6129 - accuracy: 0.6700
Epoch 2/50
167/167 [==============================] - 3s 14ms/step - loss: 0.4978 - accuracy: 0.7675
Epoch 2/50
167/167 [==============================] - 2s 11ms/step - loss: 0.4409 - accuracy: 0.8080
Epoch 3/50
167/167 [==============================] - 2s 11ms/step - loss: 0.4400 - accuracy: 0.8097
Epoch 3/50
167/167 [==============================] - 2s 11ms/step - loss: 0.4351 - accurac

...

167/167 [==============================] - 2s 10ms/step - loss: 1.1893 - accuracy: 0.7598
Epoch 100/100
167/167 [==============================] - 2s 10ms/step - loss: 1.0660 - accuracy: 0.7517
Epoch 51/100
167/167 [==============================] - 2s 10ms/step - loss: 1.5375 - accuracy: 0.7364
Epoch 52/100
167/167 [==============================] - 2s 10ms/step - loss: 0.4475 - accuracy: 0.8046
Epoch 6/100
167/167 [==============================] - 2s 10ms/step - loss: 0.6166 - accuracy: 0.7715
Epoch 54/100
167/167 [==============================] - 2s 10ms/step - loss: 0.7204 - accuracy: 0.7774
Epoch 55/100
167/167 [==============================] - 2s 10ms/step - loss: 1.0676 - accuracy: 0.7626
Epoch 52/100
167/167 [==============================] - 2s 10ms/step - loss: 1.2721 - accuracy: 0.7540
Epoch 53/100
167/167 [==============================] - 3s 12ms/step - loss: 0.4720 - accuracy: 0.7859
Epoch 2/100
167/167 [==============================] - 3s 13ms/step - loss: 0.4819 - a

/opt/anaconda3/envs/py310/lib/python3.10/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
2025-11-26 13:47:58.047785: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-11-26 13:47:58.047929: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-11-26 13:47:58.047940: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2025-11-26 13:47:58.047967: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-11-26 13:47:58.047979: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical

Epoch 1/50


2025-11-26 13:47:58.740529: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


250/250 [==============================] - 2s 3ms/step - loss: 0.5768 - accuracy: 0.7076
Epoch 2/50
250/250 [==============================] - 1s 3ms/step - loss: 0.4416 - accuracy: 0.8112
Epoch 3/50
250/250 [==============================] - 1s 3ms/step - loss: 0.4340 - accuracy: 0.8101
Epoch 4/50
250/250 [==============================] - 1s 3ms/step - loss: 0.4331 - accuracy: 0.8110
Epoch 5/50
250/250 [==============================] - 1s 3ms/step - loss: 0.4331 - accuracy: 0.8110
Epoch 6/50
250/250 [==============================] - 1s 3ms/step - loss: 0.4335 - accuracy: 0.8100
Epoch 7/50
250/250 [==============================] - 1s 3ms/step - loss: 0.4334 - accuracy: 0.8099
Epoch 8/50
250/250 [==============================] - 1s 3ms/step - loss: 0.4329 - accuracy: 0.8110
Epoch 9/50
250/250 [==============================] - 1s 3ms/step - loss: 0.4334 - accuracy: 0.8083
Epoch 10/50
250/250 [==============================] - 1s 3ms/step - loss: 0.4333 - accuracy: 0.8094
Epoch 11/5